# 03 — Run Hugging Face Evaluation

Questo notebook esegue l'inferenza sul test set utilizzando:

- il modello base `Qwen/Qwen2.5-3B-Instruct`;
- l'adapter `regulation`;
- l'adapter `positive_legends`;
- l'adapter `negative_legends`;
- l'adapter `negative_to_correction`.

Il modello base viene caricato una sola volta in 4-bit.

Le predizioni vengono salvate progressivamente in formato JSONL, in modo che
l'esecuzione possa essere ripresa in caso di interruzione.

In questo notebook non vengono calcolate metriche e non vengono interpretati
gli output prodotti dal modello.

In [1]:
from pathlib import Path
from collections import Counter
from contextlib import nullcontext
import gc
import json
import os
import random
import time

import numpy as np
import torch
import transformers
import peft

from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftModel

C:\Users\fede2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Questo notebook richiede una GPU CUDA."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2,
    ),
    "GB",
)

Transformers: 5.13.1
PEFT: 0.19.1
PyTorch: 2.6.0+cu124
CUDA disponibile: True
GPU: NVIDIA GeForce RTX 2080 SUPER
VRAM: 8.0 GB


In [3]:
PROJECT_ROOT = Path(
    r"C:\Users\fede2\Desktop\Project"
).resolve()

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "evaluation"
    / "evaluation_prompts_all.jsonl"
)

ADAPTER_PATHS = {
    "regulation": (
        PROJECT_ROOT
        / "models"
        / "adapters"
        / "regulation"
    ),

    "positive_legends": (
        PROJECT_ROOT
        / "models"
        / "adapters"
        / "positive_legends"
    ),

    "negative_legends": (
        PROJECT_ROOT
        / "models"
        / "adapters"
        / "negative_legends"
    ),

    "negative_to_correction": (
        PROJECT_ROOT
        / "models"
        / "adapters"
        / "negative_to_correction"
    ),
}

OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "predictions"
    / "hf_predictions_raw.jsonl"
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

# True: elimina eventuali predizioni precedenti.
# False: riprende l'esecuzione saltando quelle già presenti.
RESET_OUTPUT = False

# Per una prova veloce puoi impostare, ad esempio, 8.
# Per l'esecuzione completa lasciare None.
MAX_EXAMPLES = None

print("Input:", INPUT_PATH)
print("Output:", OUTPUT_PATH)

Input: C:\Users\fede2\Desktop\Project\data\processed\evaluation\evaluation_prompts_all.jsonl
Output: C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_raw.jsonl


In [4]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"File dei prompt non trovato:\n{INPUT_PATH}\n\n"
        "Eseguire prima il notebook 02."
    )

for adapter_name, adapter_path in ADAPTER_PATHS.items():
    if not adapter_path.exists():
        raise FileNotFoundError(
            f"Adapter {adapter_name!r} non trovato:\n"
            f"{adapter_path}"
        )

    adapter_config = (
        adapter_path
        / "adapter_config.json"
    )

    if not adapter_config.exists():
        raise FileNotFoundError(
            f"L'adapter {adapter_name!r} non contiene "
            f"adapter_config.json:\n{adapter_path}"
        )

print("Prompt e adapter trovati correttamente.")

Prompt e adapter trovati correttamente.


In [5]:
def read_jsonl(path: Path) -> list[dict]:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"JSON non valido alla riga {line_number}: "
                    f"{error}"
                ) from error

            records.append(record)

    return records


evaluation_records = read_jsonl(INPUT_PATH)

if MAX_EXAMPLES is not None:
    evaluation_records = evaluation_records[:MAX_EXAMPLES]

print("Esempi caricati:", len(evaluation_records))
print(
    "Distribuzione:",
    Counter(
        record["task_name"]
        for record in evaluation_records
    ),
)

Esempi caricati: 160
Distribuzione: Counter({'sst2_style': 20, 'paraphrase_style': 20, 'stsb_style': 20, 'qqp_style': 20, 'mnli_style': 20, 'rte_style': 20, 'bias_detection': 20, 'grounded_answer_generation': 20})


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

base_model.eval()

print("Modello base caricato.")
print(
    "Memoria GPU allocata:",
    round(
        torch.cuda.memory_allocated()
        / 1024**3,
        2,
    ),
    "GB",
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:09<00:00, 43.48it/s]


Modello base caricato.
Memoria GPU allocata: 1.92 GB


In [7]:
adapter_items = list(
    ADAPTER_PATHS.items()
)

first_name, first_path = adapter_items[0]

model = PeftModel.from_pretrained(
    base_model,
    first_path,
    adapter_name=first_name,
    is_trainable=False,
)

for adapter_name, adapter_path in adapter_items[1:]:
    model.load_adapter(
        adapter_path,
        adapter_name=adapter_name,
        is_trainable=False,
    )

model.eval()

print(
    "Adapter caricati:",
    list(model.peft_config.keys()),
)

Adapter caricati: ['regulation', 'positive_legends', 'negative_legends', 'negative_to_correction']


In [8]:
def append_jsonl(
    path: Path,
    record: dict,
) -> None:
    with path.open(
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


def generate_prediction(
    record: dict,
) -> tuple[str, float]:
    prompt_text = record["prompt_text"]

    generation_config = record.get(
        "generation_config",
        {},
    )

    max_new_tokens = generation_config.get(
        "max_new_tokens",
        192,
    )

    encoded = tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False,
    )

    input_device = (
        model.get_input_embeddings()
        .weight
        .device
    )

    encoded = {
        key: value.to(input_device)
        for key, value in encoded.items()
    }

    input_length = encoded[
        "input_ids"
    ].shape[1]

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    elapsed_seconds = (
        time.perf_counter()
        - start_time
    )

    new_token_ids = generated_ids[
        0,
        input_length:,
    ]

    raw_output = tokenizer.decode(
        new_token_ids,
        skip_special_tokens=True,
    ).strip()

    del encoded
    del generated_ids
    del new_token_ids

    return raw_output, elapsed_seconds

In [9]:
if RESET_OUTPUT and OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Output precedente eliminato.")


completed_predictions = set()

if OUTPUT_PATH.exists():
    existing_predictions = read_jsonl(
        OUTPUT_PATH
    )

    completed_predictions = {
        (
            record["model_name"],
            record["example_id"],
        )
        for record in existing_predictions
    }

    print(
        "Predizioni già presenti:",
        len(completed_predictions),
    )

else:
    print("Nessuna predizione precedente.")

Nessuna predizione precedente.


In [10]:
MODEL_VARIANTS = [
    ("base", None),
    ("regulation", "regulation"),
    ("positive_legends", "positive_legends"),
    ("negative_legends", "negative_legends"),
    (
        "negative_to_correction",
        "negative_to_correction",
    ),
]


for model_name, adapter_name in MODEL_VARIANTS:
    print("\n" + "=" * 80)
    print("MODELLO:", model_name)
    print("=" * 80)

    if adapter_name is None:
        adapter_context = model.disable_adapter()
    else:
        model.set_adapter(
            adapter_name,
            inference_mode=True,
        )

        adapter_context = nullcontext()

    with adapter_context:
        for record in tqdm(
            evaluation_records,
            desc=model_name,
        ):
            prediction_key = (
                model_name,
                record["example_id"],
            )

            if prediction_key in completed_predictions:
                continue

            try:
                raw_output, elapsed_seconds = (
                    generate_prediction(record)
                )

                prediction_record = {
                    "example_id": record["example_id"],
                    "task_name": record["task_name"],
                    "model_name": model_name,
                    "adapter_name": adapter_name,
                    "y_true": record["y_true"],
                    "allowed_labels": record.get(
                        "allowed_labels"
                    ),
                    "raw_output": raw_output,
                    "generation_config": record.get(
                        "generation_config",
                        {},
                    ),
                    "elapsed_seconds": round(
                        elapsed_seconds,
                        4,
                    ),
                    "prompt_sha256": record.get(
                        "prompt_metadata",
                        {},
                    ).get(
                        "prompt_sha256"
                    ),
                    "metadata": record.get(
                        "metadata",
                        {},
                    ),
                    "status": "completed",
                    "error": None,
                }

            except Exception as error:
                prediction_record = {
                    "example_id": record["example_id"],
                    "task_name": record["task_name"],
                    "model_name": model_name,
                    "adapter_name": adapter_name,
                    "y_true": record["y_true"],
                    "allowed_labels": record.get(
                        "allowed_labels"
                    ),
                    "raw_output": None,
                    "generation_config": record.get(
                        "generation_config",
                        {},
                    ),
                    "elapsed_seconds": None,
                    "prompt_sha256": record.get(
                        "prompt_metadata",
                        {},
                    ).get(
                        "prompt_sha256"
                    ),
                    "metadata": record.get(
                        "metadata",
                        {},
                    ),
                    "status": "error",
                    "error": (
                        f"{type(error).__name__}: {error}"
                    ),
                }

            append_jsonl(
                OUTPUT_PATH,
                prediction_record,
            )

            completed_predictions.add(
                prediction_key
            )

    torch.cuda.empty_cache()
    gc.collect()

print("\nInferenza terminata.")


MODELLO: base


base: 100%|██████████| 160/160 [02:16<00:00,  1.17it/s]



MODELLO: regulation


regulation: 100%|██████████| 160/160 [02:18<00:00,  1.16it/s]



MODELLO: positive_legends


positive_legends: 100%|██████████| 160/160 [02:05<00:00,  1.27it/s]



MODELLO: negative_legends


negative_legends: 100%|██████████| 160/160 [02:04<00:00,  1.29it/s]



MODELLO: negative_to_correction


negative_to_correction: 100%|██████████| 160/160 [02:10<00:00,  1.22it/s]


Inferenza terminata.


In [11]:
predictions = read_jsonl(
    OUTPUT_PATH
)

expected_predictions = (
    len(evaluation_records)
    * len(MODEL_VARIANTS)
)

completed = [
    record
    for record in predictions
    if record["status"] == "completed"
]

errors = [
    record
    for record in predictions
    if record["status"] == "error"
]

prediction_counts = Counter(
    record["model_name"]
    for record in predictions
)

print("Predizioni attese:", expected_predictions)
print("Predizioni salvate:", len(predictions))
print("Completate:", len(completed))
print("Errori:", len(errors))

print("\nDistribuzione per modello:")

for model_name, _ in MODEL_VARIANTS:
    print(
        f"- {model_name}: "
        f"{prediction_counts.get(model_name, 0)}"
    )

if len(predictions) != expected_predictions:
    print(
        "\nATTENZIONE: il numero di predizioni "
        "non coincide con quello atteso."
    )

if errors:
    print("\nPrime predizioni in errore:")

    for record in errors[:10]:
        print(
            record["model_name"],
            record["example_id"],
            record["error"],
        )

print("\nOutput:")
print(OUTPUT_PATH)

Predizioni attese: 800
Predizioni salvate: 800
Completate: 800
Errori: 0

Distribuzione per modello:
- base: 160
- regulation: 160
- positive_legends: 160
- negative_legends: 160
- negative_to_correction: 160

Output:
C:\Users\fede2\Desktop\Project\data\predictions\hf_predictions_raw.jsonl


Al termine dell’esecuzione completa devi ottenere:

Predizioni attese: 800
Predizioni salvate: 800

- base: 160
- regulation: 160
- positive_legends: 160
- negative_legends: 160
- negative_to_correction: 160

Il file prodotto sarà:

data/predictions/hf_predictions_raw.jsonl

Non viene ancora stabilito se una risposta sia corretta. Anche output come:

The answer is compliant.

oppure:

4.

restano invariati dentro raw_output. La pulizia e la conversione nelle label ufficiali spettano al successivo notebook:

04_parse_predictions.ipynb